In [5]:
import pandas as pd
import numpy as np
from scipy.special import xlogy

Loans_df = pd.read_csv(r'E:\DS\tezendra\Projects\JP Morgan Job Simulation\Task 3\Task 3 and 4_Loan_Data.csv')  

In [7]:

def compute_optimal_buckets(df, n_buckets=6, criterion='mse'):
    # Group by FICO score
    grouped = df.groupby('fico_score').agg(
        total=('default', 'count'),
        defaults=('default', 'sum')
    ).reset_index()
    grouped = grouped.sort_values('fico_score')
    
    scores = grouped['fico_score'].values
    n = grouped['total'].values
    k = grouped['defaults'].values
    M = len(scores)
    
    # Prefix sums for quick interval computations
    pref_n = np.concatenate([[0], np.cumsum(n)])
    pref_k = np.concatenate([[0], np.cumsum(k)])
    pref_s = np.concatenate([[0], np.cumsum(n * scores)])
    pref_q = np.concatenate([[0], np.cumsum(n * scores * scores)])
    
    def interval_cost(i, j):
        """Cost for bucket covering scores i..j (inclusive)."""
        N = pref_n[j+1] - pref_n[i]
        K = pref_k[j+1] - pref_k[i]
        if N == 0:
            return 0.0
        if criterion == 'mse':
            S = pref_s[j+1] - pref_s[i]
            Q = pref_q[j+1] - pref_q[i]
            return Q - (S * S) / N
        elif criterion == 'loglik':
            if K == 0 or K == N:
                return 0.0
            p = K / N
            # negative log-likelihood
            return -(K * np.log(p) + (N - K) * np.log(1 - p))
        else:
            raise ValueError("criterion must be 'mse' or 'loglik'")
    

    INF = 1e18
    dp = np.full((n_buckets + 1, M + 1), INF)
    best = np.zeros((n_buckets + 1, M + 1), dtype=int)
    dp[0][0] = 0
    
    for t in range(1, n_buckets + 1):
        for i in range(1, M + 1):
            # j is the start index of the last bucket (0-based), so j < i
            # interval [j, i-1]
            for j in range(t-1, i):  # need at least t-1 elements for previous buckets
                cost_interval = interval_cost(j, i-1)
                val = dp[t-1][j] + cost_interval
                if val < dp[t][i]:
                    dp[t][i] = val
                    best[t][i] = j
    

    boundaries_idx = []
    cur = M
    for t in range(n_buckets, 0, -1):
        start = best[t][cur]
        boundaries_idx.append(cur - 1)  # last index of this bucket
        cur = start
    # boundaries_idx is from best bucket to worst (since we backtracked)
    boundaries_idx.reverse()
    # boundaries_idx[0] is last index of worst bucket (lowest scores)
    # boundaries_idx[-1] is last index of best bucket (highest scores)
    # Convert to FICO scores: the maximum score in each bucket
    boundaries = [scores[idx] for idx in boundaries_idx]
    
    # Create rating map: for each score, assign rating based on which bucket it falls into.
    rating_map = {}
    current_rating = n_buckets  # worst rating
    prev_bound = scores[0] - 1  # start below min
    for idx, b in enumerate(boundaries):
        # bucket idx covers scores > prev_bound and <= b
        for score in scores:
            if prev_bound < score <= b:
                rating_map[score] = n_buckets - idx  # idx 0 -> rating n_buckets, idx last -> rating 1
        prev_bound = b
    

    return boundaries, rating_map


for criterion in ['mse', 'loglik']:
    boundaries, rating_map = compute_optimal_buckets(Loans_df, n_buckets=6, criterion=criterion)
    print(f"Optimal boundaries ({criterion}): {boundaries}")
    print("Rating map (sample):")
    # Show mapping for a few scores
    for score in [300, 400, 500, 600, 650, 700, 750, 800, 850]:
        if score in rating_map:
            print(f"  FICO {score} -> Rating {rating_map[score]}")
    print("-" * 50)

Optimal boundaries (mse): [np.int64(544), np.int64(595), np.int64(636), np.int64(676), np.int64(723), np.int64(850)]
Rating map (sample):
  FICO 500 -> Rating 6
  FICO 600 -> Rating 4
  FICO 650 -> Rating 3
  FICO 700 -> Rating 2
  FICO 750 -> Rating 1
  FICO 800 -> Rating 1
  FICO 850 -> Rating 1
--------------------------------------------------
Optimal boundaries (loglik): [np.int64(520), np.int64(579), np.int64(611), np.int64(649), np.int64(719), np.int64(850)]
Rating map (sample):
  FICO 500 -> Rating 6
  FICO 600 -> Rating 4
  FICO 650 -> Rating 2
  FICO 700 -> Rating 2
  FICO 750 -> Rating 1
  FICO 800 -> Rating 1
  FICO 850 -> Rating 1
--------------------------------------------------
